Nonostante l'architettura PRO, MNIST ha una varianza limitata. Per spingere il modello verso il 99.6%, dobbiamo simulare una scrittura manuale più 'sporca'.
Modifica: inserisci un layer di RandomRotation(0.1) e uno di RandomZoom(0.1) subito dopo l'input del modello.
Sperimentazione: aumenta il numero di filtri del primo blocco a 64 e osserva se il tempo di addestramento per epoca giustifica il guadagno in accuratezza
Verifica: calcola la metrice di confuzione sul test set per identificare quali coppie di cifre (es. 4 e 9) sono ancora fonte di errore per il tuo modello Pro

In [1]:
# MNIST è un dataset didattico di 60.000 immagini di training e 10.000 di test, tutte 28x28 in scala di grigi

import os

#
# 1. DEFINIZIONE DEL MOTORE DI CALCOLO (Best Practice 2026)
#
# Nel 2026, l'agnosticismo del framework è la norma. Impostiamo PyTorch come backend.
os.environ["KERAS_BACKEND"] = "torch"

import keras
#import torch # Utilizzato sotto il cofano da Keras, ma non è necessario importarlo
import numpy as np
from sklearn.metrics import confusion_matrix # Import aggiunto per la verifica finale

#verifica del backend effettivamente in uso
print("Backend Keras:", keras.config.backend())

#
# 2. CARICAMENTO E PREPARAZIONE "PRO" DEI DATI
#

# Carichiamo MNIST. Teoria: MNIST è il 'Hello World' della computer vision, 
# ma qui lo trattiamo con rigore industriale.
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
#ora ogni elemento x_train è una matrice 28x28. Ogni valore rappresenta la luminosità di un pixel (0=nero 255=bianco)
#ogni elemento y_train è invece un numero intero da 0 a 9

# Reshape: Aggiungiamo il canale colore (1 per scala di grigi) richiesto dai layer Conv2D.
# il canale aggiutno vale 1 perchè l'immagine è in scala di grigi, per un'immagine RGB avresti 3 perche i canali sono rosso, verde, blu
# (batch, altezza, larghezza, canali)
# .astype("float32") converte i numeri in decimali a 32 bit, necessario perche il modello lavora con numeri in virgola mobile
# /255.0 normalizza i valori 255->1.0   128->circa 0.502 0->0.0
# questo rende l'ottimizzatore più stabile
x_train = x_train.reshape(-1, 28, 28, 1).astype("float32") / 255.0
x_test = x_test.reshape(-1, 28, 28, 1).astype("float32") / 255.0

#
# 3. COSTRUZIONE DEL MODELLO - ARCHITETTURA ESTREMA (BLOCCHI CONV-CONV-POOL)
#

# Teoria: Usare due Conv2D prima del pooling aumenta la non-linearità e il campo ricettivo
# senza ridurre troppo velocemente la risoluzione spaziale (Feature Map preservation).
def build_mnist_pro_model():

    # Inputs
    inputs = keras.Input(shape=(28, 28, 1))

    # Blocco 0: Data Augmentation On-the-Fly (per simulare scrittura "sporca")
    x = keras.layers.RandomRotation(0.1)(inputs)
    x = keras.layers.RandomZoom(0.1)(x)    
    
    # Blocco 1: Low-level features (Sperimentazione: aumento filtri a 64 per catturare più varianza)
    x = keras.layers.Conv2D(64, (3, 3), padding="same", activation="relu")(inputs) #32=il layer impara 32 filtri; Kernel (3,3) ogni filtro osserva inizialmente una finestra di 3x3 pixel; padding='same' la convoluzione mantiene altezza e larghezza, ingresso 28x28x1 uscita 28x28x1
    x = keras.layers.Conv2D(64, (3, 3), padding="same", activation="relu")(x)
    x = keras.layers.BatchNormalization()(x) # Stabilizza il gradiente durante il backprop
    x = keras.layers.MaxPooling2D((2, 2))(x) #osserva ogni area (2,2) e conserva solo il valore massimo esemio 1,4 e 2,3 mantiene solo 4. La forma passa da 28x28x32 a 14x14x32
    # il pooling riduce il costo computazionale ma elimina anche informazione spaziale. Per questo non conviene applicarlo troppo presto e troppe volte
    x = keras.layers.Dropout(0.2)(x) # Regolarizzazione: previene la co-adattazione dei neuroni

    # Blocco 2: Mid-level features
    x = keras.layers.Conv2D(64, (3, 3), padding="same", activation="relu")(x)
    x = keras.layers.Conv2D(64, (3, 3), padding="same", activation="relu")(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.MaxPooling2D((2, 2))(x) #parto da 14,14 dopo il pooling ho 7x7, pero' con Conv2D ho aumentato i canali
    #mano mano che la risoluzione diminuisce il modello può rappresentare un numero maggiore di caratteristiche astratte (cos' non perdo informazine)
    x = keras.layers.Dropout(0.3)(x)

    # Blocco 3: High-level features & Global Context
    x = keras.layers.Conv2D(128, (3, 3), padding="same", activation="relu")(x)
    x = keras.layers.Conv2D(128, (3, 3), padding="same", activation="relu")(x)
    x = keras.layers.BatchNormalization()(x)
    
    # Teoria: Il GlobalAveragePooling2D riduce drasticamente i parametri rispetto a un layer Dense,
    # agendo come una potente regolarizzazione strutturale contro l'overfitting.
    x = keras.layers.GlobalAveragePooling2D()(x)
    
    # Classificatore finale
    x = keras.layers.Dense(128, activation="relu")(x)
    x = keras.layers.Dropout(0.4)(x)
    outputs = keras.layers.Dense(10, activation="softmax")(x)
    
    return keras.Model(inputs, outputs)

model = build_mnist_pro_model()

# 4. COMPILAZIONE E OTTIMIZZATORE
# Utilizziamo AdamW: una variante di Adam che gestisce meglio il weight decay (L2).
model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# 5. CALLBACKS AVANZATE (Controllo del Top Performance)
# Strategia: Se la loss non scende, riduciamo la velocità (LR) per 'rifinire' i pesi.
callbacks = [
    # Riduce il LR se la val_loss smette di migliorare (fine-tuning dinamico)
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6),
    # Ferma il training prima che inizi l'overfitting
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True),
    # Salvataggio atomico nel formato standard 2026 (.keras)
    keras.callbacks.ModelCheckpoint("mnist_pro_model.keras", save_best_only=True)
]

# 6. TRAINING LOOP
print("[INFO] Addestramento MNIST Pro in corso...")
history = model.fit(
    x_train, y_train,
    epochs=50, # EarlyStopping gestirà la durata reale
    batch_size=128,
    validation_split=0.1,
    callbacks=callbacks,
    verbose=1
)

# 7. VALUTAZIONE: MATRICE DI CONFUSIONE
loss, acc = model.evaluate(x_test, y_test, verbose=0)
print(f"\n[RISULTATO] Accuratezza sul Test Set: {acc*100:.2f}%")

# Calcolo Matrice di Confusione
y_pred_probs = model.predict(x_test, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

# Teoria: La matrice evidenzia se il modello "soffre" ancora su coppie ambigue (es: 4 vs 9)
cm = confusion_matrix(y_test, y_pred)
print("\n[VERIFICA] Matrice di Confusione:")
print(cm)

Backend Keras: torch
[INFO] Addestramento MNIST Pro in corso...
Epoch 1/50
422/422 ━━━━━━━━━━━━━━━━━━━━ 21s 49ms/step - accuracy: 0.9361 - loss: 0.2155 - val_accuracy: 0.6542 - val_loss: 1.1842 - learning_rate: 0.0010
Epoch 2/50
422/422 ━━━━━━━━━━━━━━━━━━━━ 21s 50ms/step - accuracy: 0.9832 - loss: 0.0565 - val_accuracy: 0.9845 - val_loss: 0.0541 - learning_rate: 0.0010
Epoch 3/50
422/422 ━━━━━━━━━━━━━━━━━━━━ 21s 50ms/step - accuracy: 0.9875 - loss: 0.0420 - val_accuracy: 0.9903 - val_loss: 0.0348 - learning_rate: 0.0010
Epoch 4/50
422/422 ━━━━━━━━━━━━━━━━━━━━ 22s 53ms/step - accuracy: 0.9895 - loss: 0.0349 - val_accuracy: 0.9790 - val_loss: 0.0689 - learning_rate: 0.0010
Epoch 5/50
422/422 ━━━━━━━━━━━━━━━━━━━━ 24s 57ms/step - accuracy: 0.9913 - loss: 0.0297 - val_accuracy: 0.9925 - val_loss: 0.0249 - learning_rate: 0.0010
Epoch 6/50
422/422 ━━━━━━━━━━━━━━━━━━━━ 24s 57ms/step - accuracy: 0.9920 - loss: 0.0267 - val_accuracy: 0.9905 - val_loss: 0.0357 - learning_rate: 0.0010
Epoch 7/50
4